# 09b — psfFlux / scienceFlux / templateFlux for Gaia DR3–matched Fink objects
# (input from data_FINK_BLOCK_LC_01)

## Purpose

This notebook is a **variant of `09_psfFlux_scienceFlux_templateFlux_GaiaDR3matching.ipynb`**:
it uses flux data from `data_FINK_BLOCK_LC_01/` instead of `data_FINK_BLOCK_LC_03/`.
It displays the raw fluxes in nJy for the subset of Fink diaObjects that were
successfully joined to a Gaia DR3 source in `08b_matchFinkAlertswithGaiaDR3.ipynb`.

The three flux quantities compared are:

- `psfFlux`      — difference-image PSF flux (can be negative)
- `scienceFlux`  — PSF flux on the science calexp (positive for real sources)
- `templateFlux` — PSF flux on the template image

**No normalisation** is applied.  The Gaia DR3 match provides additional
context: Gaia G magnitude, stability flag, and Fink group are shown in the
figure titles.

## Flux data source

Two Gaia-star subsets from `data_FINK_BLOCK_LC_01/` are **concatenated** to
recover the full set of Gaia-matched diaObjects:

| Parquet file | Gaia category | `gaia_origin` tag |
|--------------|---------------|-------------------|
| `gaia_star_stable_hq_src.parquet` | HQ stars with astrometric parallax solution | `hq` |
| `gaia_nophotgstar_stable_unknown_parallax_src.parquet` | Stable stars without reliable parallax | `unknownparallax` |

Note: unlike `data_FINK_BLOCK_LC_03/`, the parquet files in `_01/` do not carry
a `_joined_butler` / `_joined_consdb` suffix — they are single files per subset.

Both subsets are photometrically stable (`gaia_STABLE = True`) and equally
valid for photometric calibration purposes.

## Inputs

| File | Produced by |
|------|-------------|
| `data_FINK_BLOCK_LC_01/gaia_star_stable_hq_src.parquet` | Fink alert pipeline (block 01) |
| `data_FINK_BLOCK_LC_01/gaia_nophotgstar_stable_unknown_parallax_src.parquet` | Fink alert pipeline (block 01) |
| `data_FINK_BLOCK_LC_01/gaia_star_stable_hq_fp.parquet` | Fink alert pipeline (block 01) |
| `data_FINK_BLOCK_LC_01/gaia_nophotgstar_stable_unknown_parallax_fp.parquet` | Fink alert pipeline (block 01) |
| `data_FINK_BLOCK_LC_08b/fink_diaobj_gaia_join_matched.csv` | `08b_matchFinkAlertswithGaiaDR3.ipynb` |

---

- **Author:** Sylvie Dagoret-Campagne  
- **Affiliation:** IJCLab / IN2P3 / CNRS  
- **Follows:** `09_psfFlux_scienceFlux_templateFlux_GaiaDR3matching.ipynb` and `08b_matchFinkAlertswithGaiaDR3.ipynb`
- **Creation date:** 2026-04-15
- **Last update:** 2026-04-15  
- **Subject:** Fink alert broker — Rubin LSST DIA photometry diagnostics on Gaia DR3 stars (block 01 inputs)


## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_09b"

# Flux data directory — block 01 (no butler/consdb split, single file per subset)
DIR_DATA_FLUX = "data_FINK_BLOCK_LC_01"

# Two Gaia-stable subsets to concatenate:
#   hq              → stars with a reliable astrometric parallax solution
#   unknownparallax → stable stars without a parallax solution (equally good photometrically)
# Note: in _01/ the parquet files are NOT split by butler/consdb — single file per subset.
FILE_SRC_HQ = os.path.join(DIR_DATA_FLUX, "gaia_star_stable_hq_src.parquet")
FILE_SRC_UNK = os.path.join(DIR_DATA_FLUX, "gaia_nophotgstar_stable_unknown_parallax_src.parquet")

# fp (forced-photometry / diaObject-level) files for reference flux lines
FILE_FP_HQ = os.path.join(DIR_DATA_FLUX, "gaia_star_stable_hq_fp.parquet")
FILE_FP_UNK = os.path.join(DIR_DATA_FLUX, "gaia_nophotgstar_stable_unknown_parallax_fp.parquet")

# Gaia DR3 matched catalogue from notebook 08b (expected: 131 diaObjects matched)
DIR_DATA_GAIA08b = "data_FINK_BLOCK_LC_08b"
FILE_GAIA_MATCHED = os.path.join(DIR_DATA_GAIA08b, "fink_diaobj_gaia_join_matched.csv")

DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)

N_TOP = 131  # keep all Gaia-matched objects

BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

# ── Column names (Fink alert schema) ─────────────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_SRC = "r:diaSourceId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_PSF = "r:psfFlux"
COL_PSFERR = "r:psfFluxErr"

SCIENCE_FLUX_CANDIDATES = [
    "r:scienceFlux",
    "r:psfScience",
    "r:psfScienceFlux",
    "scienceFlux",
    "psfScience",
    "psfScienceFlux",
]
SCIENCE_ERR_CANDIDATES = [
    "r:scienceSigma",
    "r:scienceFluxErr",
    "r:psfScienceSigma",
    "r:psfScienceFluxErr",
    "scienceSigma",
    "scienceFluxErr",
    "psfScienceSigma",
]
TEMPLATE_FLUX_CANDIDATES = [
    "psfTemplateFlux",
    "r:templateFlux",
    "r:template",
    "templateFlux",
    "template",
]
TEMPLATE_ERR_CANDIDATES = [
    "r:templateSigma",
    "r:templateFluxErr",
    "r:psfTemplateSigma",
    "r:psfTemplateFluxErr",
    "templateSigma",
    "templateFluxErr",
    "psfTemplateSigma",
]


def savefig(name):
    """Save current figure as PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Flux data dir    : {os.path.abspath(DIR_DATA_FLUX)}")
print(f"Gaia match file  : {os.path.abspath(FILE_GAIA_MATCHED)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"N_TOP            : {N_TOP}")

## 2. Load flux data (src parquet from data_FINK_BLOCK_LC_01)

We concatenate **two** Gaia-stable subsets:

- `gaia_star_stable_hq_src`  — stars with a reliable parallax (`gaia_origin = 'hq'`)
- `gaia_nophotgstar_stable_unknown_parallax_src` — stable stars without reliable parallax (`gaia_origin = 'unknownparallax'`)

Unlike `data_FINK_BLOCK_LC_03/`, block 01 parquet files are **not split** into
`_butler` / `_consdb` variants — each subset is a single parquet file.
The `gaia_origin` column is added **before** concatenation so that the provenance
of each row is always traceable.

In [ ]:
def _load_single(path, origin_tag):
    """
    Load a single parquet file and tag all rows with `gaia_origin = origin_tag`.
    Returns DataFrame or None if the file does not exist.
    """
    if os.path.exists(path):
        df = pd.read_parquet(path)
        df["gaia_origin"] = origin_tag  # provenance column
        print(f"  Loaded [{origin_tag:18s}] : {path}  ({len(df):,} rows)")
        return df
    else:
        print(f"  WARNING: file not found for origin='{origin_tag}': {path}")
        return None


print("Loading HQ src parquet …")
df_hq = _load_single(FILE_SRC_HQ, "hq")

print("Loading unknown_parallax src parquet …")
df_unk = _load_single(FILE_SRC_UNK, "unknownparallax")

# Concatenate non-None frames
frames = [f for f in [df_hq, df_unk] if f is not None]
if len(frames) == 0:
    raise FileNotFoundError(
        "No flux parquet files found in "
        + DIR_DATA_FLUX
        + "\nExpected files:\n"
        + f"  {FILE_SRC_HQ}\n"
        + f"  {FILE_SRC_UNK}"
    )

df_src = pd.concat(frames, ignore_index=True)

# Derive a single src_label for file-naming
if df_hq is not None and df_unk is not None:
    src_label = "hq_plus_unknownparallax"
elif df_hq is not None:
    src_label = "hq_only"
else:
    src_label = "unknownparallax_only"

print(f"\nCombined shape    : {df_src.shape[0]:,} rows × {df_src.shape[1]} columns")
print(f"src_label         : {src_label}")
print(f"gaia_origin counts:\n{df_src['gaia_origin'].value_counts().to_string()}")

## 3. Load Gaia DR3 matched catalogue (from notebook 08b)

This catalogue contains one row per diaObject with the Gaia DR3 `source_id`
already joined.  We use it to:

1. Build the list of `diaObjectId` values that have a confirmed Gaia DR3 match.
2. Attach Gaia metadata (G magnitude, Fink group, Gaia stability class) to the
   figure titles for scientific context.

In [ ]:
if not os.path.exists(FILE_GAIA_MATCHED):
    raise FileNotFoundError(
        f"{FILE_GAIA_MATCHED} not found.\nRun notebook 08b_matchFinkAlertswithGaiaDR3.ipynb first."
    )

df_gaia_matched = pd.read_csv(FILE_GAIA_MATCHED)

print("Gaia matched catalogue shape:", df_gaia_matched.shape)
print("Columns:", df_gaia_matched.columns.tolist())
print(df_gaia_matched.head(3).to_string())

In [ ]:
# Build lookup dict: diaObjectId → Gaia metadata row
# We index on the 'diaObjectId' column (plain integer, not the Fink 'r:diaObjectId' with prefix)
GAIA_ID_COL = "diaObjectId"

if GAIA_ID_COL not in df_gaia_matched.columns:
    raise KeyError(
        f"Column '{GAIA_ID_COL}' not found in {FILE_GAIA_MATCHED}.\n"
        f"Available columns: {df_gaia_matched.columns.tolist()}"
    )

# Set of Gaia-matched diaObjectIds (as strings for safe comparison)
gaia_matched_ids = set(df_gaia_matched[GAIA_ID_COL].astype(str).values)
print(f"Unique Gaia-matched diaObjectIds: {len(gaia_matched_ids)}")

# Lookup dict for per-object Gaia metadata
df_gaia_matched_idx = df_gaia_matched.set_index(GAIA_ID_COL)


def get_gaia_meta(obj_id):
    """Return a dict with Gaia metadata for a given diaObjectId, or empty dict."""
    key = int(str(obj_id))
    if key in df_gaia_matched_idx.index:
        row = df_gaia_matched_idx.loc[key]
        return {
            "group": row.get("group", "?"),
            "field": row.get("field", "?"),
            "G_mag": row.get("gaia_phot_g_mean_mag", np.nan),
            "gaia_stable": row.get("gaia_STABLE", None),
            "gaia_variable": row.get("gaia_is_gaia_variable", None),
            "gaia_status": row.get("gaia_status", "?"),
            "nDiaSrc": row.get("nDiaSources", None),
            "dr3_id": row.get("dr3_source_id", None),
        }
    return {}


print("Lookup function get_gaia_meta() defined.")

## 4. Schema inspection — detect scienceFlux and templateFlux columns

In [ ]:
df = df_src  # alias used throughout this notebook

COL_SCI = None
COL_SCIERR = None
for candidate in SCIENCE_FLUX_CANDIDATES:
    if candidate in df.columns:
        COL_SCI = candidate
        print(f"Found science flux column  : '{COL_SCI}'")
        break
for candidate in SCIENCE_ERR_CANDIDATES:
    if candidate in df.columns:
        COL_SCIERR = candidate
        print(f"Found science flux err col : '{COL_SCIERR}'")
        break

COL_TEMP = None
COL_TEMPERR = None
for candidate in TEMPLATE_FLUX_CANDIDATES:
    if candidate in df.columns:
        COL_TEMP = candidate
        print(f"Found template flux column  : '{COL_TEMP}'")
        break
for candidate in TEMPLATE_ERR_CANDIDATES:
    if candidate in df.columns:
        COL_TEMPERR = candidate
        print(f"Found template flux err col : '{COL_TEMPERR}'")
        break

if COL_SCI is None:
    print("WARNING: No scienceFlux column found.")
if COL_TEMP is None:
    print("WARNING: No templateFlux column found.")

has_science = COL_SCI is not None
has_sci_err = COL_SCIERR is not None
has_template = COL_TEMP is not None
has_template_err = COL_TEMPERR is not None

print(f"\n  psfFlux      : {COL_PSF}")
print(f"  scienceFlux  : {COL_SCI}")
print(f"  scienceErr   : {COL_SCIERR}")
print(f"  templateFlux : {COL_TEMP}")
print(f"  templateErr  : {COL_TEMPERR}")

## 5. Filter flux data to Gaia DR3–matched diaObjects

We keep only the rows of the (concatenated) flux parquet whose `r:diaObjectId`
appears in the Gaia-matched catalogue produced by notebook 08b.
The `gaia_origin` column is preserved so that downstream analyses can
distinguish `hq` from `unknownparallax` rows.

In [ ]:
# Cast parquet object column to string for safe set membership test
df["_oid_str"] = df[COL_OBJ].astype(str)

n_before = len(df)
df_gaia = df[df["_oid_str"].isin(gaia_matched_ids)].copy()
n_after = len(df_gaia)

print(f"Alerts before Gaia filter : {n_before:,}")
print(f"Alerts after  Gaia filter : {n_after:,}  ({100 * n_after / n_before:.1f}%)")
print(f"Unique diaObjects kept    : {df_gaia[COL_OBJ].nunique()}")
print()
print("gaia_origin breakdown among Gaia-matched alerts:")
print(df_gaia["gaia_origin"].value_counts().to_string())

## 6. Rank Gaia-matched objects by alert count

Sort diaObjects by decreasing number of alerts and keep the top N_TOP.
The `gaia_origin` tag is reported in the ranking table.

In [ ]:
# Count alerts per diaObject and rank
counts = df_gaia.groupby(COL_OBJ).size().reset_index(name="n_alerts")
counts = counts.sort_values("n_alerts", ascending=False).reset_index(drop=True)

# Attach gaia_origin per object (take the first value — all rows for the same object share the same tag)
origin_map = df_gaia.groupby(COL_OBJ)["gaia_origin"].first()
counts["gaia_origin"] = counts[COL_OBJ].map(origin_map)

# Attach Gaia G magnitude for context
counts["G_mag"] = counts[COL_OBJ].apply(lambda x: get_gaia_meta(x).get("G_mag", np.nan))
counts["gaia_status"] = counts[COL_OBJ].apply(lambda x: get_gaia_meta(x).get("gaia_status", "?"))

top_objects = counts.head(N_TOP)[COL_OBJ].tolist()
df_top = df_gaia[df_gaia[COL_OBJ].isin(top_objects)].copy()

print(f"Total Gaia-matched objects  : {len(counts)}")
print(f"Top N_TOP = {N_TOP} objects selected")
print()
print(counts.head(20).to_string(index=False))

## 7. Load fp catalogue from data_FINK_BLOCK_LC_01 (for median flux reference lines)

In [ ]:
# Optional: load the fp (forced-photometry / diaObject-level) parquet files to draw
# per-band mean flux reference lines in the per-object plots.
# In data_FINK_BLOCK_LC_01/ these are single files (no butler/consdb split).
CAT_PATHS = [
    FILE_FP_HQ,
    FILE_FP_UNK,
]

cat_frames = []
for cp in CAT_PATHS:
    if os.path.exists(cp):
        try:
            cat_frames.append(pd.read_parquet(cp))
            print(f"  Loaded fp catalogue: {cp}")
        except Exception as e:
            print(f"  Could not read {cp}: {e}")

if cat_frames:
    df_cat_ref = pd.concat(cat_frames, ignore_index=True)
    # Deduplicate on diaObjectId (keep first occurrence)
    if "diaObjectId" in df_cat_ref.columns:
        df_cat_ref = df_cat_ref.drop_duplicates(subset="diaObjectId")
    print(f"  Combined fp catalogue: {len(df_cat_ref):,} rows")
else:
    df_cat_ref = None
    print("  No fp catalogue found — reference flux lines will be skipped.")

## 7b. Helper functions

In [ ]:
def mjd_to_datestr(mjd_array):
    """Convert an array of MJD values to ISO date strings (YYYY-MM-DD)."""
    return [Time(m, format="mjd").to_datetime().strftime("%Y-%m-%d") for m in mjd_array]


def flux_to_mag(flux_nJy, zp_ab=31.4):
    """Convert flux in nJy to AB magnitude.  Returns (mag, nan) or (nan, nan)."""
    if flux_nJy is None or not np.isfinite(flux_nJy) or flux_nJy <= 0:
        return np.nan, np.nan
    mag = -2.5 * np.log10(flux_nJy) + zp_ab
    return mag, np.nan


def _shared_lim(arrays, margin=0.05):
    """Compute shared [vmin, vmax] across a list of arrays, with a fractional margin."""
    flat = np.concatenate([a.ravel() for a in arrays if a is not None and len(a) > 0])
    flat = flat[np.isfinite(flat)]
    if len(flat) == 0:
        return None
    vmin, vmax = flat.min(), flat.max()
    span = max(vmax - vmin, 1e-10)
    return [vmin - margin * span, vmax + margin * span]


def _add_date_axis(ax, dt_days, t0_mjd, n_ticks=4):
    """Add a secondary x-axis showing calendar dates above the panel."""
    xlims = ax.get_xlim()
    tick_pos = np.linspace(xlims[0], xlims[1], n_ticks)
    tick_mjds = tick_pos + t0_mjd
    tick_labels = [Time(m, format="mjd").to_datetime().strftime("%m/%d") for m in tick_mjds]
    ax2 = ax.twiny()
    ax2.set_xlim(xlims)
    ax2.set_xticks(tick_pos)
    ax2.set_xticklabels(tick_labels, fontsize=6, rotation=30, ha="left")


def _object_color_palette(n):
    """Return a list of n distinct colours using tab20 + tab20b."""
    c1 = [cm.tab20(i) for i in range(20)]
    c2 = [cm.tab20b(i) for i in range(20)]
    palette = (c1 + c2) * (n // 40 + 1)
    return palette[:n]


print("Helper functions defined.")

## 8. Per-object flux plots

One 7-panel figure per Gaia-matched diaObject (u g r i z y | all bands).
The `gaia_origin` tag (`hq` or `unknownparallax`) is shown in the figure title
alongside the Gaia G magnitude and stability class.

In [ ]:
def plot_fluxes_one_object(obj_id, df_obj, src_label, rank):
    """
    Plot psfFlux, scienceFlux, and templateFlux for one Gaia-matched DIA object.

    Layout: 7 panels (u g r i z y | all-bands combined).
    Shared x and y scales across the 6 band panels.
    Figure title includes Gaia DR3 metadata and gaia_origin provenance.
    """

    # Retrieve gaia_origin for this object
    origin = df_obj["gaia_origin"].iloc[0] if "gaia_origin" in df_obj.columns else "?"

    # Optional catalogue reference flux
    oid_str = str(obj_id)
    ref_psf = {}
    ref_sci = {}
    if df_cat_ref is not None:
        mask_cat = df_cat_ref["diaObjectId"].astype(str) == oid_str
        if mask_cat.any():
            cat_row = df_cat_ref[mask_cat].iloc[0]
            for b in BANDS:
                for dest, col in [(ref_psf, f"r:{b}_psfFluxMean"), (ref_sci, f"r:{b}_scienceFluxMean")]:
                    try:
                        fval = float(cat_row[col])
                        dest[b] = fval if np.isfinite(fval) else None
                    except (KeyError, TypeError, ValueError):
                        dest[b] = None

    meta = get_gaia_meta(obj_id)
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]
    field = meta.get("field", df_obj["field"].iloc[0] if "field" in df_obj.columns else "")
    group = meta.get("group", "?")
    G_mag = meta.get("G_mag", np.nan)
    gaia_status = meta.get("gaia_status", "?")

    n_subplots = len(BANDS) + 1
    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 4.5),
        constrained_layout=True,
    )

    band_data = {}
    all_flux = []
    all_dt = []

    for band in BANDS:
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)
        if len(df_b) == 0:
            band_data[band] = None
            continue
        dt = df_b[COL_MJD].values - t0_mjd
        psf = df_b[COL_PSF].values.astype(float)
        psf_err = df_b[COL_PSFERR].values.astype(float)
        sci = df_b[COL_SCI].values.astype(float) if has_science else None
        sci_err = df_b[COL_SCIERR].values.astype(float) if has_sci_err else None
        temp = df_b[COL_TEMP].values.astype(float) if has_template else None
        temp_err = df_b[COL_TEMPERR].values.astype(float) if has_template_err else None

        band_data[band] = (dt, psf, psf_err, sci, sci_err, temp, temp_err)
        all_dt.append(dt)
        all_flux.append(psf)
        if sci is not None:
            all_flux.append(sci)
        if temp is not None:
            all_flux.append(temp)

    ylim = _shared_lim(all_flux) if all_flux else None
    xlim = _shared_lim(all_dt) if all_dt else None

    for idx, band in enumerate(BANDS):
        ax = axes[idx]
        color = BAND_COLORS.get(band, "k")

        if band_data[band] is None:
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9)
            if xlim:
                ax.set_xlim(xlim)
            if ylim:
                ax.set_ylim(ylim)
            ax.set_xlabel("\u0394t [days]", fontsize=8)
            continue

        dt, psf, psf_err, sci, sci_err, temp, temp_err = band_data[band]

        if sci is not None:
            ax.errorbar(
                dt,
                sci,
                yerr=sci_err,
                fmt="o",
                ms=5,
                color=color,
                ecolor=color,
                elinewidth=0.9,
                capsize=2,
                alpha=0.90,
                label="scienceFlux",
            )

        ax.errorbar(
            dt,
            psf,
            yerr=psf_err,
            fmt="s",
            ms=4,
            color=color,
            ecolor=color,
            elinewidth=0.7,
            capsize=2,
            alpha=0.55,
            mfc="none",
            label="psfFlux",
        )

        if temp is not None:
            ax.errorbar(
                dt,
                temp,
                yerr=temp_err,
                fmt="+",
                ms=5,
                color=color,
                ecolor=color,
                elinewidth=0.7,
                capsize=2,
                alpha=0.55,
                mfc="none",
                label="templateFlux",
            )

        v_sci = ref_sci.get(band)
        if v_sci is not None:
            mag, _ = flux_to_mag(v_sci)
            ax.axhline(v_sci, color=color, lw=1.2, ls="-.", alpha=0.85, label=f"obj: m={mag:.1f} mag")
        v_psf = ref_psf.get(band)
        if v_psf is not None:
            mag, _ = flux_to_mag(v_psf)
            ax.axhline(v_psf, color=color, lw=1.2, ls=":", alpha=0.85, label=f"diaobj: m={mag:.1f} mag")

        ax.axhline(0.0, color="k", lw=0.6, ls=":", alpha=0.4)
        ax.set_title(f"band {band}", fontsize=9)
        ax.set_ylabel("flux [nJy]", fontsize=8)
        ax.set_xlabel("\u0394t [days]", fontsize=8)
        ax.legend(fontsize=7, loc="best")
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        _add_date_axis(ax, dt, t0_mjd)

    # Combined panel
    ax_all = axes[-1]
    for band in BANDS:
        if band_data[band] is None:
            continue
        dt, psf, psf_err, sci, sci_err, temp, temp_err = band_data[band]
        color = BAND_COLORS.get(band, "k")
        if sci is not None:
            ax_all.errorbar(
                dt,
                sci,
                yerr=sci_err,
                fmt="o",
                ms=3,
                color=color,
                ecolor=color,
                elinewidth=0.7,
                capsize=2,
                alpha=0.80,
                label=f"{band} sci",
            )
        ax_all.errorbar(
            dt,
            psf,
            yerr=psf_err,
            fmt="s",
            ms=3,
            color=color,
            ecolor=color,
            elinewidth=0.6,
            capsize=2,
            alpha=0.45,
            mfc="none",
            label=f"{band} psf",
        )
        if temp is not None:
            ax_all.errorbar(
                dt,
                temp,
                yerr=temp_err,
                fmt="+",
                ms=3,
                color=color,
                ecolor=color,
                elinewidth=0.7,
                capsize=2,
                alpha=0.80,
                mfc="none",
                label=f"{band} temp",
            )
        v_sci = ref_sci.get(band)
        if v_sci is not None:
            ax_all.axhline(v_sci, color=color, lw=0.7, ls="-.", alpha=0.45, label="_nolegend_")
        v_psf = ref_psf.get(band)
        if v_psf is not None:
            ax_all.axhline(v_psf, color=color, lw=0.7, ls=":", alpha=0.45, label="_nolegend_")

    ax_all.axhline(0.0, color="k", lw=0.6, ls=":", alpha=0.4)
    ax_all.set_title("all bands \u2014 sci(\u25cf) psf(\u25a1) temp(+)", fontsize=9)
    ax_all.set_ylabel("flux [nJy]", fontsize=8)
    ax_all.set_xlabel("\u0394t [days]", fontsize=8)
    ax_all.legend(fontsize=6, ncol=4, loc="best")
    if xlim:
        ax_all.set_xlim(xlim)
    _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

    G_str = f"{G_mag:.2f}" if np.isfinite(G_mag) else "?"
    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field}  |  N={len(df_obj)}  |  t\u2080={t0_date}\n"
        f"Fink group: {group}   |   Gaia G={G_str} mag   |   Gaia status: {gaia_status}   |   origin: {origin}\n"
        "scienceFlux (\u25cf) vs psfFlux (\u25a1) vs templateFlux (+)   [nJy, no normalisation]",
        fontsize=10,
        y=1.09,
    )
    savefig(f"fluxes_gaia_obj_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()


print("plot_fluxes_one_object() defined.")

In [ ]:
if not has_science:
    print("scienceFlux column not available — Section 8 skipped.")
else:
    for rank, obj_id in enumerate(top_objects):
        df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
        plot_fluxes_one_object(obj_id, df_obj, src_label, rank)

## 9. Per-band multi-object overview

One figure per band (+ combined): all top-N Gaia-matched objects plotted on
the same axes with a distinct colour per object.

In [ ]:
obj_colors = _object_color_palette(len(top_objects))

for band_sel in BANDS + ["all"]:
    fig, ax = plt.subplots(figsize=(11, 4), constrained_layout=True)

    for o_idx, obj_id in enumerate(top_objects):
        meta = get_gaia_meta(obj_id)
        group = meta.get("group", "?")
        G_str = f"G={meta.get('G_mag', np.nan):.1f}" if np.isfinite(meta.get("G_mag", np.nan)) else "G=?"
        obj_color = obj_colors[o_idx]

        df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
        t0_mjd = df_obj[COL_MJD].min()

        if band_sel == "all":
            df_b = df_obj
        else:
            df_b = df_obj[df_obj[COL_BAND] == band_sel]
        if len(df_b) == 0:
            continue

        dt = df_b[COL_MJD].values.astype(float) - t0_mjd
        psf = df_b[COL_PSF].values.astype(float)
        psf_err = df_b[COL_PSFERR].values.astype(float)
        sci = df_b[COL_SCI].values.astype(float) if has_science else None
        sci_err = df_b[COL_SCIERR].values.astype(float) if has_sci_err else None

        label_base = f"obj{o_idx + 1} {G_str} ({group})"

        if sci is not None:
            ax.errorbar(
                dt,
                sci,
                yerr=sci_err,
                fmt="o",
                ms=3,
                color=obj_color,
                ecolor=obj_color,
                elinewidth=0.6,
                capsize=1.5,
                alpha=0.85,
                label=label_base + " sci",
            )
        ax.errorbar(
            dt,
            psf,
            yerr=psf_err,
            fmt="s",
            ms=2.5,
            color=obj_color,
            ecolor=obj_color,
            elinewidth=0.5,
            capsize=1,
            alpha=0.45,
            mfc="none",
            label=label_base + " psf",
        )

    ax.axhline(0.0, color="k", lw=0.6, ls=":", alpha=0.4)
    band_lbl = band_sel if band_sel != "all" else "all bands"
    ax.set_title(
        f"Gaia-matched objects \u2014 {band_lbl}  |  scienceFlux (\u25cf) vs psfFlux (\u25a1)  [nJy]",
        fontsize=10,
    )
    ax.set_xlabel("\u0394t [days from object t\u2080]", fontsize=9)
    ax.set_ylabel("flux [nJy]", fontsize=9)
    ax.legend(fontsize=6, ncol=4, loc="best")

    fname = f"multiobj_band{band_sel}_{src_label}"
    savefig(fname)
    plt.show()

## 10. Summary statistics: median flux levels

For each Gaia-matched object and band, report the median `scienceFlux`,
`psfFlux`, and `templateFlux`, plus the ratio
`median(psfFlux) / median(scienceFlux)`.  This should be ~0 for a stable
star whose template was built from images of the same star.

The `gaia_origin` column is included so results can be split by provenance.

In [ ]:
records = []

for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id]
    meta = get_gaia_meta(obj_id)
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]
    origin = df_obj["gaia_origin"].iloc[0] if "gaia_origin" in df_obj.columns else "?"

    row = {
        "rank": rank + 1,
        "diaObjectId": obj_id,
        "n_alerts": len(df_obj),
        "t0_date": t0_date,
        "gaia_origin": origin,
        "fink_group": meta.get("group", "?"),
        "gaia_G_mag": meta.get("G_mag", np.nan),
        "gaia_status": meta.get("gaia_status", "?"),
        "dr3_source_id": meta.get("dr3_id", None),
    }

    for band in BANDS:
        df_b = df_obj[df_obj[COL_BAND] == band]
        if len(df_b) == 0:
            row[f"med_sci_{band}"] = np.nan
            row[f"med_psf_{band}"] = np.nan
            row[f"med_temp_{band}"] = np.nan
            row[f"ratio_psf_sci_{band}"] = np.nan
            continue

        med_psf = float(np.nanmedian(df_b[COL_PSF].values.astype(float)))
        row[f"med_psf_{band}"] = round(med_psf, 2)

        if has_science:
            med_sci = float(np.nanmedian(df_b[COL_SCI].values.astype(float)))
            row[f"med_sci_{band}"] = round(med_sci, 2)
            row[f"ratio_psf_sci_{band}"] = round(med_psf / med_sci, 4) if med_sci != 0 else np.nan
        else:
            row[f"med_sci_{band}"] = np.nan
            row[f"ratio_psf_sci_{band}"] = np.nan

        if has_template:
            row[f"med_temp_{band}"] = round(float(np.nanmedian(df_b[COL_TEMP].values.astype(float))), 2)
        else:
            row[f"med_temp_{band}"] = np.nan

    records.append(row)

df_stats = pd.DataFrame(records)
print("Median flux levels per Gaia-matched object and band [nJy]:")
print("  med_sci   = median(scienceFlux)")
print("  med_psf   = median(psfFlux)")
print("  med_temp  = median(templateFlux)")
print("  ratio     = med_psf / med_sci  (\u223c0 expected for stable star in good template)")
print("  gaia_origin = 'hq' or 'unknownparallax'")
print()
display(df_stats)

In [ ]:
out_path = os.path.join(DIR_FIGS, f"median_flux_stats_gaia_{src_label}.csv")
df_stats.to_csv(out_path, index=False)
print(f"Saved \u2192 {out_path}")

## 11. Summary

This notebook is a **variant of `09_psfFlux_scienceFlux_templateFlux_GaiaDR3matching.ipynb`**
using `data_FINK_BLOCK_LC_01/` as the flux data source instead of `data_FINK_BLOCK_LC_03/`.

Key differences vs `09`:

- `DIR_DATA_FLUX = "data_FINK_BLOCK_LC_01"` (instead of `_03`)
- Parquet files in `_01/` are **single files per subset** (no `_butler` / `_consdb` suffix):
  `gaia_star_stable_hq_src.parquet` and `gaia_nophotgstar_stable_unknown_parallax_src.parquet`
- `_load_pair()` replaced by `_load_single()` (simpler, no fallback logic needed)
- fp reference files: `gaia_star_stable_hq_fp.parquet` and
  `gaia_nophotgstar_stable_unknown_parallax_fp.parquet`
- Output figures and CSV written to `figs_FINK_BLOCK_LC_09b/`

The Gaia DR3 matched catalogue (`data_FINK_BLOCK_LC_08b/fink_diaobj_gaia_join_matched.csv`)
is shared with notebook `09`.

| Section | Content |
|---------|----------|
| 2       | Load and concatenate hq + unknownparallax src parquets from `_01/` |
| 3       | Load Gaia matched catalogue from 08b |
| 5       | Filter parquet to Gaia-matched diaObjectIds |
| 6       | Rank by alert count (with gaia_origin) |
| 8       | Per-object flux plots (7 panels each, origin in title) |
| 9       | Per-band multi-object overview |
| 10      | Summary statistics table (CSV, with gaia_origin column) |

**Outputs:** figures in `figs_FINK_BLOCK_LC_09b/` and CSV stats table.
